# Лабораторная работа №2. Повышение разрешения изображений с помощью автокодировщика

**Цель:** по монохромному изображению 7×7 пикселей восстановить изображение 28×28. Для обучения и тестирования используется Fashion MNIST. Результат сравнивается с бикубической интерполяцией.

Метрики сравнения: MSE и MAE (меньше — лучше), PSNR и SSIM (больше — лучше).

## 1. Подключение библиотек

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)

## 2. Загрузка и подготовка Fashion MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

x_train_low = tf.image.resize(x_train, (7, 7), method="area").numpy()
x_test_low = tf.image.resize(x_test, (7, 7), method="area").numpy()

print("Обучающие изображения:", x_train.shape)
print("Уменьшенные обучающие изображения:", x_train_low.shape)
print("Тестовые изображения:", x_test.shape)
print("Уменьшенные тестовые изображения:", x_test_low.shape)

Fashion MNIST содержит изображения предметов одежды в оттенках серого размером 28×28. Значения пикселей нормализуются в диапазон [0, 1]. Для получения входных данных изображения уменьшаются до 7×7 методом area. Исходные изображения 28×28 используются как целевые значения.

In [ ]:
class_names = [
    "Футболка", "Брюки", "Свитер", "Платье", "Пальто",
    "Сандалии", "Рубашка", "Кроссовки", "Сумка", "Ботинки"
]

plt.figure(figsize=(12, 4))
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("28×28")

    plt.subplot(2, 8, i + 9)
    plt.imshow(x_train_low[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("7×7")

plt.suptitle("Исходные и уменьшенные изображения Fashion MNIST")
plt.tight_layout()
plt.show()

## 3. Построение автокодировщика

Сверточные слои извлекают признаки из изображения 7×7. Транспонированные свертки последовательно увеличивают разрешение до 14×14 и 28×28. Выходной слой sigmoid формирует яркости пикселей в диапазоне [0, 1].

In [ ]:
input_layer = layers.Input(shape=(7, 7, 1))

x = layers.Conv2D(64, 3, padding="same", activation="relu")(input_layer)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)

x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)

x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)

output_layer = layers.Conv2D(1, 3, padding="same", activation="sigmoid")(x)

model = models.Model(input_layer, output_layer)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)
model.summary()

## 4. Обучение модели

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
    )
]

history = model.fit(
    x_train_low,
    x_train,
    validation_split=0.1,
    epochs=40,
    batch_size=128,
    callbacks=callbacks,
    shuffle=True
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Обучение")
plt.plot(history.history["val_loss"], label="Валидация")
plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.title("Изменение ошибки в процессе обучения")
plt.legend()
plt.grid()
plt.show()

## 5. Восстановление изображений и бикубическая интерполяция

In [ ]:
predictions = model.predict(x_test_low, batch_size=256, verbose=1)

bicubic = tf.image.resize(x_test_low, (28, 28), method="bicubic").numpy()
bicubic = np.clip(bicubic, 0.0, 1.0)

print("Автокодировщик:", predictions.shape)
print("Бикубическая интерполяция:", bicubic.shape)

## 6. Визуальное сравнение

In [ ]:
num_images = 8
plt.figure(figsize=(14, 2.2 * num_images))

for i in range(num_images):
    images = [
        x_test[i].squeeze(),
        x_test_low[i].squeeze(),
        predictions[i].squeeze(),
        bicubic[i].squeeze()
    ]
    titles = ["Оригинал 28×28", "Вход 7×7", "Автокодировщик", "Бикубическая"]

    for j, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(num_images, 4, i * 4 + j + 1)
        plt.imshow(img, cmap="gray", vmin=0, vmax=1)
        plt.axis("off")
        if i == 0:
            plt.title(title)
        if j == 0:
            plt.ylabel(class_names[y_test[i]])

plt.tight_layout()
plt.show()

## 7. Количественная оценка качества

In [ ]:
def calculate_metrics(reference, reconstructed):
    reference = tf.convert_to_tensor(reference, dtype=tf.float32)
    reconstructed = tf.convert_to_tensor(reconstructed, dtype=tf.float32)

    return {
        "MSE": float(tf.reduce_mean(tf.square(reference - reconstructed)).numpy()),
        "MAE": float(tf.reduce_mean(tf.abs(reference - reconstructed)).numpy()),
        "PSNR (dB)": float(tf.reduce_mean(
            tf.image.psnr(reference, reconstructed, max_val=1.0)
        ).numpy()),
        "SSIM": float(tf.reduce_mean(
            tf.image.ssim(reference, reconstructed, max_val=1.0)
        ).numpy())
    }

ae_metrics = calculate_metrics(x_test, predictions)
bicubic_metrics = calculate_metrics(x_test, bicubic)

print("Автокодировщик:")
for metric, value in ae_metrics.items():
    print(f"{metric}: {value:.6f}")

print("\nБикубическая интерполяция:")
for metric, value in bicubic_metrics.items():
    print(f"{metric}: {value:.6f}")

In [ ]:
metric_names = list(ae_metrics.keys())
ae_values = [ae_metrics[name] for name in metric_names]
bicubic_values = [bicubic_metrics[name] for name in metric_names]

positions = np.arange(len(metric_names))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(positions - width / 2, ae_values, width, label="Автокодировщик")
plt.bar(positions + width / 2, bicubic_values, width, label="Бикубическая интерполяция")
plt.xticks(positions, metric_names)
plt.title("Сравнение качества реконструкции")
plt.legend()
plt.grid(axis="y")
plt.show()

## 8. Ошибка на отдельных тестовых изображениях

In [ ]:
sample_count = 10

ae_sample_mse = np.mean(
    (x_test[:sample_count] - predictions[:sample_count]) ** 2,
    axis=(1, 2, 3)
)
bicubic_sample_mse = np.mean(
    (x_test[:sample_count] - bicubic[:sample_count]) ** 2,
    axis=(1, 2, 3)
)

plt.figure(figsize=(10, 4))
plt.plot(range(sample_count), ae_sample_mse, marker="o", label="Автокодировщик")
plt.plot(range(sample_count), bicubic_sample_mse, marker="s", label="Бикубическая")
plt.xlabel("Номер изображения")
plt.ylabel("MSE")
plt.title("Ошибка реконструкции отдельных изображений")
plt.legend()
plt.grid()
plt.show()

## 9. Вывод

В работе был реализован алгоритм повышения разрешения изображений Fashion MNIST с помощью сверточного автокодировщика. На вход модели подавались монохромные изображения 7×7 пикселей, а целевыми значениями являлись исходные изображения 28×28 пикселей.

Для сравнения использовалась бикубическая интерполяция, применённая к тем же уменьшенным тестовым изображениям. Качество реконструкции оценивалось по метрикам MSE, MAE, PSNR и SSIM.

После запуска ноутбука необходимо сравнить полученные значения: меньшие MSE и MAE означают меньшую ошибку, большие PSNR и SSIM — лучшее структурное и визуальное сходство. Окончательный вывод о преимуществе одного из методов следует сделать по фактическим результатам эксперимента.